## Bermudan swaption pricing

### 1.1 Classic imports

In [1]:
import numpy as np

### 1.2 Global model parameters 

In [2]:
DELTA = 0.25              # tenor spacing between reset/payment dates
T_SWAP_END = 4.0          # final swap maturity T_N
N_FORWARDS = int(T_SWAP_END / DELTA)   # number of forward rates L_1,...,L_16

# Flat initial forward curve: all forwards are 5%
INIT_FORWARDS = np.full(N_FORWARDS, 0.05)

# Volatility levels sigma_k (time-independent base level), k=1..16
VOL_SIGMA = np.array([0.20] * 7 + [0.22] * 4 + [0.24] * 5)

# Angles: theta_k = (pi/2) * (k-1)/15, k=1..16
ANGLES = 0.5 * np.pi * (np.arange(1, N_FORWARDS + 1) - 1) / 15.0

# Correlation matrix: rho_{ij} = cos(theta_i - theta_j)
CORR = np.cos(ANGLES[:, None] - ANGLES[None, :])

# Bermudan swaption setup


T_EX_START = 1.0          # first exercise date
T_EX_END   = 2.0          # last exercise date
SWAP_STRIKE = 0.05        # strike of the underlying swap

# Monte Carlo configuration for Andersen's method
N_PATHS_THRESH = 10_000   # number of paths used to determine thresholds
N_PATHS_PRICE  = 10_000   # number of paths used for pricing, given thresholds
THRESH_GRID    = np.linspace(0.0, 0.20, 5001)  # candidate exercise thresholds

N_TIME_STEPS = int(T_EX_END / DELTA)  # discrete times: 0, DELTA, ..., T_EX_END

# Fix RNG seed for reproducibility
np.random.seed(1234)

### 1.3 Time-dependent volatility tapering g_k(t)

In [3]:
def g_factor(time, k, dt=DELTA):
    """
      g_k(t) = min( ((T_k - t)^+) / (T_k - T_{k-1}), 1 )
    with T_k = k*dt and k=1..N_FORWARDS.
    """
    T_km1 = (k - 1) * dt
    T_k   = k * dt

    if time >= T_k:
        return 0.0
    elif time >= T_km1:
        return (T_k - time) / (T_k - T_km1)  # = (T_k - t)/dt
    else:
        return 1.0


def g_vector(time, n_forwards=N_FORWARDS, dt=DELTA):
    return np.array([g_factor(time, k, dt) for k in range(1, n_forwards + 1)])


### 1.4 LMM simulation under LR spot measure 

In [4]:
def simulate_all_paths(initial_curve,
                       n_forwards,
                       n_steps,
                       dt,
                       vol_vector,
                       angle_vec,
                       corr_matrix,
                       n_paths):
    """
    Simulate Libor forward rates on the time grid 0, dt, ..., T_EX_END
    under the LR spot measure.

    Returns:
        paths: array of shape (n_paths, n_steps + 1, n_forwards),
               where paths[p, t, k] is the forward rate for period [T_k, T_{k+1}] observed at time T_t

    """
    # Brownian increments (2 driving factors) for all paths and steps
    normals = np.sqrt(dt) * np.random.normal(size=(n_paths, n_steps, 2))

    all_paths = []
    for p in range(n_paths):
        path = simulate_single_path(
            initial_curve,
            n_forwards,
            n_steps,
            dt,
            vol_vector,
            angle_vec,
            corr_matrix,
            normals[p],
        )
        all_paths.append(path)
    return np.array(all_paths)


def simulate_single_path(initial_curve,
                         n_forwards,
                         n_steps,
                         dt,
                         vol_vector,
                         angle_vec,
                         corr_matrix,
                         normals_one_path):
    """
    Simulate one path of the entire forward curve.
    Returns an array of shape (n_steps+1, n_forwards).
    """
    path = [initial_curve]

    for t_idx in range(1, n_steps + 1):
        prev_curve = path[-1]
        new_curve = one_time_step_LMM(
            prev_curve,
            t_idx,
            dt,
            vol_vector,
            angle_vec,
            corr_matrix,
            n_forwards,
            normals_one_path[t_idx - 1],
        )
        path.append(new_curve)

    return np.array(path)


def one_time_step_LMM(prev_curve,
                      t_idx,
                      dt,
                      vol_vector,
                      angle_vec,
                      corr_matrix,
                      n_forwards,
                      normal_pair):
    """
    One lognormal Euler step from T_{t_idx-1} to T_{t_idx} in the LR spot measure,
    with time-dependent volatilities:
        sigma_k(t) = VOL_SIGMA[k] * g_k(t).
    """
    # Time at which the step starts
    current_time = (t_idx - 1) * dt

    # Effective volatilities sigma_k(t) = sigma_k * g_k(t)
    g_vec = g_vector(current_time, n_forwards, dt)
    sigma_eff = vol_vector * g_vec

    # Drift ingredients: dt * sigma_k(t) * L / (1 + dt * L)
    with np.errstate(invalid="ignore"):
        drift_pre = dt * sigma_eff * prev_curve / (1.0 + dt * prev_curve)

    # Two independent Brownian increments
    dW1, dW2 = normal_pair
    dWk = np.cos(angle_vec) * dW1 + np.sin(angle_vec) * dW2

    new_curve = np.full(n_forwards, np.nan)

    # Forwards with index < t_idx are already fixed; we leave them as NaN
    for k in range(t_idx, n_forwards):
        sigma_k = sigma_eff[k]
        if np.isnan(prev_curve[k]) or sigma_k == 0.0:
            new_curve[k] = prev_curve[k]
            continue

        contrib = corr_matrix[k] * drift_pre
        # nansum ignores NaN contributions (including index 0 if needed)
        mu_k = sigma_k * np.nansum(contrib[t_idx - 1 : k + 1])

        dWk_k = dWk[k]
        new_curve[k] = prev_curve[k] * np.exp(
            (mu_k - 0.5 * sigma_k**2) * dt + sigma_k * dWk_k
        )

    return new_curve

### 1.5 Pricing primitives : bonds, swap, LR discount factor 

In [5]:
def zero_coupon(forward_matrix, dt, t_idx, k_idx):
    """
    Zero-coupon bond price P(T_t, T_k) with t_idx < k_idx.

    forward_matrix: 2D array [time_index, forward_index].
    """
    row = forward_matrix[t_idx]
    # Factors 1 + L_{j+1}(T_t)*dt for j = t_idx,...,k_idx-1
    factors = (1.0 + dt * row)[t_idx:k_idx]
    return 1.0 / np.prod(factors)


def payer_swap_value(forward_matrix, dt, n_forwards, t_idx, strike):
    """
    Value at time T_t of a payer swap (pay float, receive fixed) with
    maturity T_N. This is used to construct the receiver swaption payoff.
    """
    row = forward_matrix[t_idx]

    bond_vals = np.array([
        zero_coupon(forward_matrix, dt, t_idx, k) if k > t_idx else np.nan
        for k in range(1, n_forwards + 1)
    ])

    # Approximation: dt * sum_k P(T_t, T_k) * (L_k(T_t) - K)
    return dt * np.nansum(bond_vals * (row - strike))


def lr_discount_factor(forward_matrix, dt, t_idx):
    """
    Stochastic discount factor from T_0 to T_t under LR spot measure.

    We use the diagonal entries L_{j+1}(T_j), j=0,...,t_idx-1.
    """
    diag_fwd = np.diagonal(forward_matrix)[:t_idx]
    return 1.0 / np.prod(1.0 + dt * diag_fwd)

### 1.6 Andersen Bermudan algorithm 

In [6]:
def bermudan_andersen(
    paths_for_thresholds,
    paths_for_pricing,
    dt,
    n_forwards,
    threshold_grid,
    T_start,
    T_end,
    strike,
):
    """
    Complete Andersen procedure:

      (1) Use the first set of Monte Carlo paths to determine the
          optimal exercise thresholds on each exercise date.
      (2) Use a fresh set of paths and these thresholds to estimate
          the Bermudan swaption price.

    Returns:
        price_0, thresholds_list
        - thresholds_list is ordered from first to last exercise date.
    """
    start_idx = int(T_start / dt)
    end_idx   = int(T_end   / dt)

    # At final exercise date we take a trivial threshold (never exercise later).
    thresholds = [0.0]

    # Phase 1: find thresholds on the first set of paths
    _ = andersen_pass(
        simulated_paths=paths_for_thresholds,
        dt=dt,
        n_forwards=n_forwards,
        threshold_grid=threshold_grid,
        start_idx=start_idx,
        end_idx=end_idx,
        strike=strike,
        thresholds=thresholds,
        find_thresholds=True,
    )

    # Phase 2: price the Bermudan using the calibrated thresholds
    price = andersen_pass(
        simulated_paths=paths_for_pricing,
        dt=dt,
        n_forwards=n_forwards,
        threshold_grid=threshold_grid,
        start_idx=start_idx,
        end_idx=end_idx,
        strike=strike,
        thresholds=thresholds,
        find_thresholds=False,
    )

    # Reorder thresholds so that index 0 corresponds to the earliest exercise date
    thresholds.reverse()
    return price, thresholds


def andersen_pass(
    simulated_paths,
    dt,
    n_forwards,
    threshold_grid,
    start_idx,
    end_idx,
    strike,
    thresholds,
    find_thresholds,
):
    """
    One backward pass of Andersen's algorithm.

    If find_thresholds is True, thresholds are calibrated by grid search.
    Otherwise, the pre-calibrated thresholds are used to compute the price.
    """
    n_paths = len(simulated_paths)

    # Terminal condition at the last exercise date: trivial threshold = 0
    terminal_payoff = np.zeros(n_paths)

    discounted_values = andersen_step(
        simulated_paths=simulated_paths,
        dt=dt,
        n_forwards=n_forwards,
        n_paths=n_paths,
        threshold_grid=threshold_grid,
        end_idx=end_idx,
        strike=strike,
        discounted_future=terminal_payoff,
        thresholds=thresholds,
        time_idx=end_idx,
        find_thresholds=False,  # never optimize threshold at the last date
    )

    # Backward induction from end_idx-1 down to start_idx
    for t_idx in range(end_idx - 1, start_idx - 1, -1):
        discounted_values = andersen_step(
            simulated_paths=simulated_paths,
            dt=dt,
            n_forwards=n_forwards,
            n_paths=n_paths,
            threshold_grid=threshold_grid,
            end_idx=end_idx,
            strike=strike,
            discounted_future=discounted_values,
            thresholds=thresholds,
            time_idx=t_idx,
            find_thresholds=find_thresholds,
        )

    return np.mean(discounted_values)


def andersen_step(
    simulated_paths,
    dt,
    n_forwards,
    n_paths,
    threshold_grid,
    end_idx,
    strike,
    discounted_future,
    thresholds,
    time_idx,
    find_thresholds,
):
    """
    Single backward step of Andersen's algorithm at a given exercise date.
    """
    # Immediate exercise value in LR numéraire:
    # receiver swaption on a payer swap -> payoff = - value of the payer swap
    exercise_val = np.array([
        -payer_swap_value(simulated_paths[i], dt, n_forwards, time_idx, strike)
        for i in range(n_paths)
    ])

    # LR spot measure stochastic discount factor to T_0
    discounts = np.array([
        lr_discount_factor(simulated_paths[i], dt, time_idx)
        for i in range(n_paths)
    ])

    if find_thresholds:
        # For each candidate threshold, compute the corresponding mean payoff
        prices = [
            apply_threshold(exercise_val, discounts, discounted_future, thr, True)
            for thr in threshold_grid
        ]
        # Select the threshold that maximizes the mean payoff
        best_thr = threshold_grid[int(np.argmax(prices))]
        thresholds.append(best_thr)
    else:
        # Use thresholds already determined in the first pass (stored backwards)
        best_thr = thresholds[end_idx - time_idx]

    # Update discounted payoff using the chosen threshold
    return apply_threshold(
        exercise_val, discounts, discounted_future, best_thr, False
    )


def apply_threshold(
    exercise_val,
    discounts,
    discounted_future,
    threshold,
    return_mean,
):
    """
    Given a threshold c on the exercise value, we adopt the rule:
        exercise if exercise_val > c, otherwise continue.

    If return_mean is True, return the average discounted payoff.
    Otherwise, return the pathwise discounted payoffs.
    """
    continue_mask = exercise_val <= threshold

    result = discounts * exercise_val
    result[continue_mask] = discounted_future[continue_mask]

    if return_mean:
        return np.mean(result)
    return result


### 1.7 Results

In [7]:
print("=== Bermudan swaption: simulation ===")

paths_thresh_final = simulate_all_paths(
    initial_curve=INIT_FORWARDS,
    n_forwards=N_FORWARDS,
    n_steps=N_TIME_STEPS,
    dt=DELTA,
    vol_vector=VOL_SIGMA,
    angle_vec=ANGLES,
    corr_matrix=CORR,
    n_paths=N_PATHS_THRESH,
)

paths_price_final = simulate_all_paths(
    initial_curve=INIT_FORWARDS,
    n_forwards=N_FORWARDS,
    n_steps=N_TIME_STEPS,
    dt=DELTA,
    vol_vector=VOL_SIGMA,
    angle_vec=ANGLES,
    corr_matrix=CORR,
    n_paths=N_PATHS_PRICE,
)

price_final, thresholds_final = bermudan_andersen(
    paths_for_thresholds=paths_thresh_final,
    paths_for_pricing=paths_price_final,
    dt=DELTA,
    n_forwards=N_FORWARDS,
    threshold_grid=THRESH_GRID,
    T_start=T_EX_START,
    T_end=T_EX_END,
    strike=SWAP_STRIKE,
)

print("\nEstimated Bermudan receiver swaption price at t = 0 :")
print(price_final)

print("\nEstimated optimal exercise thresholds (on exercise value):")
exercise_dates = np.arange(T_EX_START, T_EX_END + 1e-9, DELTA)
for t, thr in zip(exercise_dates, thresholds_final):
    print(f"  at T = {t:.2f}  ->  threshold ≈ {thr}")


=== Bermudan swaption: simulation ===

Estimated Bermudan receiver swaption price at t = 0 :
0.013816462775811581

Estimated optimal exercise thresholds (on exercise value):
  at T = 1.00  ->  threshold ≈ 0.022840000000000003
  at T = 1.25  ->  threshold ≈ 0.0164
  at T = 1.50  ->  threshold ≈ 0.01404
  at T = 1.75  ->  threshold ≈ 0.00808
  at T = 2.00  ->  threshold ≈ 0.0
